In [ ]:
from io import StringIO
import pandas as pd
import requests
import time

# This script is best run in the background because it takes approximately 15 minutes to run.
# It's a lot of data you're pulling! 

# ----------------------- RECORD DATES ----------------------- #
thom_temp_start_of_record = "2019-08-02T00:00:00Z"             #
thom_disch_start_of_record = "1990-10-01T00:00:00Z"            #
thom_id = "USGS-01184000"                                      #
                                                               #
had_temp_start_of_record = "2010-09-09T00:00:00Z"              #
had_disch_start_of_record = "2008-12-05T00:00:00Z"             #
haddam_id = "USGS-01193050"                                    #
                                                               #
hol_disch_start_of_record = "2002-11-02T00:00:00Z"             #
hol_id = "USGS-01172010"                                       #

mont_disch_start_of_record = "1993-04-18T00:00:00Z"
mont_id = "USGS-01170500"
# ------------------------------------------------------------ #

# Parameter codes:
discharge_code = "00060"
temp_code = "00010"

url_info = {
"base_url": "https://api.waterdata.usgs.gov/ogcapi/v0/collections/continuous/items?f=csv&lang=en-US&limit=50000&properties=monitoring_location_id,parameter_code,time,value,unit_of_measure&skipGeometry=true&offset=0&monitoring_location_id=",
"url_stage2": "&parameter_code=",
"url_stage3": "&time=",
"header": {"X-Api-Key":"1r3zYYned5vnRusLPI5nJXIHBe2iam3eYxjwaZqi"} # TODO: YOU MUST PLACE YOUR API KEY FROM USGS INSIDE EMPTY QUOTES.
}                          # TODO: You can sign up at https://api.waterdata.usgs.gov/signup/
                           # TODO: If you use without an API key, you may be rate limited (rate limit is 1hr wait!)

def get_mean_df(site, parameter, start_of_record, url_info): # This function gets all the existing data for a variable and site, and creates a master dataframe with a mean of each day's values
    todaydt = pd.Timestamp.now(tz='UTC').normalize()
    today = todaydt.strftime("%Y-%m-%dT%H:%M:%SZ")

    day_difference = (pd.to_datetime(today[:10]) - pd.to_datetime(start_of_record[:10])).days

    url = f"{url_info["base_url"]}{site}{url_info["url_stage2"]}{parameter}{url_info["url_stage3"]}" # Assemble URL

    dflist = list()
    
    for i in range(0, day_difference, 150): # Loops through every day since start of record to current day in 550 day increments instead of max 1100 to avoid truncated data (max return values are 50,000)
        time.sleep(3)
        date1 = pd.to_datetime(start_of_record) + pd.Timedelta(days=i)
        date2 = date1 + pd.Timedelta(days = 150)
        if date2 > todaydt:
            date2 = todaydt
        
        response = requests.get(f"{url}{date1.strftime("%Y-%m-%dT%H:%M:%SZ")}/{(date2 - pd.Timedelta(minutes=5)).strftime("%Y-%m-%dT%H:%M:%SZ")}", headers=url_info["header"])
        while response.status_code == 429:
            # Check if 'Retry-After' header exists
            if response.headers.get("Retry-After"): # Not sure this code works... but hopefully it's not necessary
                wait_time = int(response.headers.get("Retry-After"))
                print(f"Rate limited. Waiting for {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                time.sleep(60)
            response = requests.get(f"{url}{date1.strftime("%Y-%m-%dT%H:%M:%SZ")}/{(date2 - pd.Timedelta(minutes=5)).strftime("%Y-%m-%dT%H:%M:%SZ")}", headers=url_info["header"])

        if not response.text.strip():
            print(f"Warning: Received empty data from API: {date1}/{date2}, {parameter}, {site}")
            df = pd.DataFrame()
        else:
            df = pd.read_csv(StringIO(response.text), sep=',')

        if "x" in df.columns:
            df.drop(columns=["x"], inplace=True, errors="ignore")
        if "y" in df.columns:
            df.drop(columns=["y"], inplace=True, errors="ignore")
            
        dflist.append(df)
        if date2 == todaydt:
            break

    fulldf = pd.DataFrame()
    fulldf = pd.concat([fulldf] + dflist, ignore_index=True)
    fulldf["time"] = pd.to_datetime(fulldf["time"], utc=True, errors="coerce")
    fulldf["value"] = pd.to_numeric(fulldf["value"], errors="coerce")

    fulldf["time"] = fulldf["time"].dt.tz_convert("US/Eastern")

    fulldf.dropna(inplace=True)
    fulldf.drop_duplicates(inplace=True)
    fulldf.set_index("time", inplace=True)
    fulldf.sort_index(inplace=True)
    if parameter == "00060":
        fulldf = fulldf[fulldf["value"] > 0].copy() # We don't want discharges less than 0, likely unchecked sensor errors

    fulldf = fulldf.resample("D").agg({ # This takes the mean of every day
        'monitoring_location_id': 'first',
        'parameter_code': 'first',
        'value': 'mean',
        'unit_of_measure': 'first'
    })

    fulldf.reset_index(inplace=True)
    fulldf["day_of_year"] = fulldf["time"].dt.day_of_year # Add day of year column
    units = fulldf.at[1,"unit_of_measure"]
    fulldf = fulldf.rename(columns={"value":f"value ({units})", "time":"date"}) # Remove redundant unit_of_measure column and rename value col appropriately
    fulldf.drop(columns=["unit_of_measure"], errors="ignore", inplace=True)
    fulldf.dropna(inplace=True)
    fulldf.drop_duplicates(inplace=True)
    fulldf["date"] = fulldf["date"].dt.tz_localize(None)

    time.sleep(60) # Abundance of caution to avoid timeouts, since this function gets thousands of days of data

    return fulldf

In [ ]:
# Core functionality is the cell above. More sites and parameters may be added if desired. 

In [ ]:
had_disch_means = get_mean_df(haddam_id, discharge_code, had_disch_start_of_record, url_info)
#If timeout happens, rerun this cell after 60 minutes.

In [ ]:
thom_disch_means = get_mean_df(thom_id, discharge_code, thom_disch_start_of_record, url_info)
#If timeout happens, rerun this cell after 60 minutes.

In [ ]:
hol_disch_means = get_mean_df(hol_id, discharge_code, hol_disch_start_of_record, url_info)
#If timeout happens, rerun this cell after 60 minutes.

In [ ]:
mont_disch_means = get_mean_df(mont_id, discharge_code, mont_disch_start_of_record, url_info)

In [ ]:
discharge_means = pd.concat([had_disch_means, thom_disch_means, hol_disch_means, mont_disch_means], ignore_index=True)
discharge_means.to_csv("Discharge Means.csv", index=False)

time.sleep(150) # Another abundance of caution thing. Just leaving the script in the background anyway

In [ ]:
had_temp_means = get_mean_df(haddam_id, temp_code, had_temp_start_of_record, url_info)
#If timeout happens, rerun this cell after 60 minutes.

In [ ]:
thom_temp_means = get_mean_df(thom_id, temp_code, thom_temp_start_of_record, url_info)
#If timeout happens, rerun this cell after 60 minutes.

In [ ]:
temp_means = pd.concat([had_temp_means, thom_temp_means], ignore_index=True)
temp_means.to_csv("Temp Means.csv", index=False)